# Laboratory practical: Sentinel-2 Level-1C atmospheric correction with ACOLITE (DSF)

**Goal:** Download a Sentinel-2 MSI Level-1C scene from Copernicus Data Space, perform **Dark Spectrum Fitting (DSF)** atmospheric correction with **ACOLITE**, and compare top-of-atmosphere (TOA) and atmospherically corrected (AC) reflectance.

**Outputs:** Sentinel-2 `.SAFE` input; ACOLITE `L1R` (including `rhot_*`) and `L2R` (including `rhos_*`) NetCDF files; comparison figure (PNG), and optionally `L2W` water products. This practical does **not** remove clouds.

**Prerequisites:** Internet access; a free [Copernicus Data Space account](https://dataspace.copernicus.eu/); JupyterLab; several GB of available storage. Initial ACOLITE runs may download radiative-transfer lookup tables (LUTs). Full scenes can require substantial RAM and disk space.

**Important:** Satellite Level-1C is TOA reflectance, while ACOLITE `L2R` is atmospherically corrected reflectance. The comparison uses `rhot_*` and `rhos_*` datasets at approximately the same wavelength. `rhos` is **not automatically equivalent to** water-leaving remote-sensing reflectance `Rrs`.


## 1. Understand the Sentinel-2 processing levels

| Product level | Description | Practical relevance |
|---|---|---|
| Level-0 | Raw instrument data | Not ordinarily distributed for end-user analysis |
| Level-1A / 1B | Intermediate calibrated/geometry processing products | Not normally used for this exercise |
| **Level-1C** | Orthorectified, calibrated **top-of-atmosphere** reflectance in a cartographic grid | **Required input to ACOLITE** |
| Level-2A | Bottom-of-atmosphere surface reflectance, commonly processed by Sen2Cor | Already corrected by another processor; **do not use as input for the standard L1C→DSF workflow** |

**Why correct the atmosphere?** Light detected by a satellite contains atmospheric scattering and absorption contributions as well as signals reflected/emitted by the surface. The atmospheric correction estimates and compensates for atmospheric effects to recover an estimate of surface reflectance. For coastal and inland waters, weak water-leaving signals make this particularly important.

**Why DSF?** ACOLITE's Dark Spectrum Fitting method estimates aerosol properties using darker spectral observations in an image. Its performance depends on scene conditions; cloud, sun glint, land-adjacency effects, and optically complex water can cause residual biases.


## 2. Install ACOLITE and create the Jupyter environment (one time)

Open a **terminal** (Anaconda Prompt on Windows). Follow the current [ACOLITE installation instructions](https://github.com/acolite/acolite). Typical starting commands:

```bash
git clone --depth 1 https://github.com/acolite/acolite.git
cd acolite
conda env create -f environment.yml
conda activate acolite
python -m pip install jupyterlab ipykernel requests pandas matplotlib xarray netCDF4
python -m ipykernel install --user --name acolite --display-name "Python (ACOLITE)"
jupyter lab
```

The `environment.yml` and environment name depend on the release; if the repository instructions differ, follow them. **Select the `Python (ACOLITE)` kernel** in Jupyter. If ACOLITE is installed elsewhere, set its directory below. You can use this notebook on a remote Linux workstation; no graphical ACOLITE interface is needed.


## 3. Define study area, dates, and folders

Example region is the **Kerala coast near Thiruvananthapuram**. Coordinates are ordered as **west, south, east, north** (longitude/latitude in degrees). Change the area and date interval for your own project. A satellite scene may cover a much larger area than the bounding box; ACOLITE can crop processing to your area of interest (AOI).

**Note:** The catalogue cloud-cover percentage refers to the whole scene, not necessarily your AOI. This filter helps select scenes, but does not mask or remove clouds.

In [1]:
from pathlib import Path

ROOT = Path.home() / "sentinel2_acolite_practical"
DATA_DIR = ROOT / "input_l1c"
OUTPUT_DIR = ROOT / "acolite_output"
FIG_DIR = ROOT / "figures"
for folder in (DATA_DIR, OUTPUT_DIR, FIG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

WEST, SOUTH, EAST, NORTH = 76.65, 8.25, 77.15, 8.75
START_DATE, END_DATE = "2025-12-01", "2026-03-01"
MAX_CLOUD_PERCENT = 30
ACOLITE_DIR = Path.home() / "acolite"  # Edit if installed elsewhere.
print(ROOT)

/Users/rshunmugapandi/sentinel2_acolite_practical


## 4. Search the Copernicus catalogue for Level-1C images

The OData catalogue search does not require credentials. This query selects Sentinel-2, product type **S2MSI1C**, acquisitions within the date interval, scenes intersecting the AOI, and a maximum scene-cloud percentage. It sorts recent acquisitions first. Inspect the returned product names before downloading.

Source: [Copernicus Data Space OData API](https://documentation.dataspace.copernicus.eu/APIs/OData.html).

In [2]:
import requests
import pandas as pd

CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
polygon = (f"POLYGON(({WEST} {SOUTH},{EAST} {SOUTH},{EAST} {NORTH},"
           f"{WEST} {NORTH},{WEST} {SOUTH}))")
filter_text = (
    "Collection/Name eq 'SENTINEL-2' "
    "and Attributes/OData.CSC.StringAttribute/any(a:a/Name eq 'productType' "
    "and a/OData.CSC.StringAttribute/Value eq 'S2MSI1C') "
    f"and ContentDate/Start ge {START_DATE}T00:00:00.000Z "
    f"and ContentDate/Start lt {END_DATE}T00:00:00.000Z "
    f"and OData.CSC.Intersects(area=geography'SRID=4326;{polygon}') "
    "and Attributes/OData.CSC.DoubleAttribute/any(a:a/Name eq 'cloudCover' "
    f"and a/OData.CSC.DoubleAttribute/Value le {MAX_CLOUD_PERCENT})"
)
params = {"$filter": filter_text, "$top": 20, "$orderby": "ContentDate/Start desc"}
response = requests.get(CATALOGUE, params=params, timeout=90)
response.raise_for_status()
products = pd.DataFrame(response.json().get("value", []))
if products.empty:
    raise RuntimeError("No L1C scenes found. Widen dates or cloud-cover limit.")
display(products[["Name", "Id", "ContentDate", "ContentLength"]].head(10))

,Name,Id,ContentDate,ContentLength
0,S2C_MSIL1C_20260226T050801_N0512_R019_T43PFK_2...,0d45070c-1e86-4432-aaa6-04f7355c1f85,"{'Start': '2026-02-26T05:08:01.025000Z', 'End'...",600339538
1,S2C_MSIL1C_20260226T050801_N0512_R019_T43PGK_2...,eaa17ebe-02e5-4f52-86d6-78f1c9d04335,"{'Start': '2026-02-26T05:08:01.025000Z', 'End'...",761391435
2,S2B_MSIL1C_20260221T050729_N0512_R019_T43PFK_2...,5206c464-8e2c-4688-99f0-c95e6eb691ba,"{'Start': '2026-02-21T05:07:29.024000Z', 'End'...",667773083
3,S2B_MSIL1C_20260211T050839_N0512_R019_T43PFK_2...,c4634097-8594-402b-a9cb-be0d1735f922,"{'Start': '2026-02-11T05:08:39.024000Z', 'End'...",604483275
4,S2C_MSIL1C_20260206T051011_N0512_R019_T43PGK_2...,8d1a6148-94ac-46b5-82a9-2c65f7dfcf03,"{'Start': '2026-02-06T05:10:11.025000Z', 'End'...",812230933
5,S2C_MSIL1C_20260206T051011_N0512_R019_T43PFK_2...,c1dc4812-cac5-45a0-8cae-5b71e06bc389,"{'Start': '2026-02-06T05:10:11.025000Z', 'End'...",616198670
6,S2B_MSIL1C_20260201T050929_N0511_R019_T43PFK_2...,217eaaff-cb53-4ec0-bfba-0de35b96199e,"{'Start': '2026-02-01T05:09:29.024000Z', 'End'...",615230385
7,S2B_MSIL1C_20260122T051019_N0511_R019_T43PFK_2...,bed5bc93-55ec-4f50-a2e6-35cfe0846ce6,"{'Start': '2026-01-22T05:10:19.024000Z', 'End'...",609571490
8,S2B_MSIL1C_20260122T051019_N0511_R019_T43PGK_2...,f4d25a1b-6d91-4a40-b657-965d2d28e565,"{'Start': '2026-01-22T05:10:19.024000Z', 'End'...",787275730
9,S2C_MSIL1C_20260117T051141_N0511_R019_T43PGK_2...,fee23938-7b69-45ac-a048-fcafaa5bde09,"{'Start': '2026-01-17T05:11:41.025000Z', 'End'...",797672161


## 5. Choose a scene and download it

The default is the first returned product. Change `SCENE_INDEX` to choose another row. Downloads require a **Copernicus Data Space username and password**. They are entered securely using hidden prompts rather than written into the notebook. The ZIP file may be hundreds of megabytes or larger. Do not interrupt the transfer.

If your account uses an authentication method that does not permit password-grant tokens, consult the current CDSE authentication guidance for an appropriate client/flow. Alternatively, download an L1C `.SAFE` product manually from Copernicus Browser and skip to **Step 7**.

In [3]:
SCENE_INDEX = 0
selected = products.iloc[SCENE_INDEX]
PRODUCT_ID, PRODUCT_NAME = selected["Id"], selected["Name"]
ZIP_PATH = DATA_DIR / f"{PRODUCT_NAME.removesuffix('.SAFE')}.zip"
print("Selected:", PRODUCT_NAME)
print("Size (GB):", round(int(selected["ContentLength"]) / 1e9, 2))

Selected: S2C_MSIL1C_20260226T050801_N0512_R019_T43PFK_20260226T084516.SAFE
Size (GB): 0.6


In [4]:
from getpass import getpass

username = input("CDSE username/email: ").strip()
password = getpass("CDSE password (hidden): ")
token_response = requests.post(
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
    data={"client_id": "cdse-public", "grant_type": "password",
          "username": username, "password": password},
    timeout=60,
)
token_response.raise_for_status()
access_token = token_response.json()["access_token"]
del password
print("Authenticated.")

CDSE username/email:  rebysam@gmail.com
CDSE password (hidden):  ········


Authenticated.


In [ ]:
DOWNLOAD_URL = f"https://download.dataspace.copernicus.eu/odata/v1/Products({PRODUCT_ID})/$value"
if not ZIP_PATH.exists():
    with requests.get(DOWNLOAD_URL, headers={"Authorization": f"Bearer {access_token}"},
                      stream=True, timeout=(30, 180), allow_redirects=True) as r:
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
print("Downloaded:", ZIP_PATH, "Size (GB):", round(ZIP_PATH.stat().st_size / 1e9, 2))

## 6. Extract the downloaded Level-1C archive

The extracted `.SAFE` directory contains Sentinel-2 metadata and JPEG 2000 image bands. ACOLITE reads the SAFE product directly. If you already have an extracted SAFE folder, **skip extraction** and set `SAFE_PATH` to that directory in Step 7.

In [6]:
import zipfile

with zipfile.ZipFile(ZIP_PATH) as zf:
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"Corrupt archive member: {bad}")
    for member in zf.infolist():
        target = (DATA_DIR / member.filename).resolve()
        if not target.is_relative_to(DATA_DIR.resolve()):
            raise ValueError("Unsafe ZIP member path")
    zf.extractall(DATA_DIR)
print("Extracted SAFE folders:", len(list(DATA_DIR.glob("*.SAFE"))))

Extracted SAFE folders: 1


## 7. Select input and verify ACOLITE

ACOLITE must be imported from its source directory in the chosen kernel. This also checks that an L1C SAFE folder exists. If you downloaded data manually, update `SAFE_PATH` and leave the earlier download steps unrun.

In [7]:
import sys

SAFE_PATH = DATA_DIR / PRODUCT_NAME  # Or: Path("/path/to/your/S2...SAFE")
assert SAFE_PATH.is_dir(), f"SAFE directory missing: {SAFE_PATH}"
assert (ACOLITE_DIR / "acolite").is_dir(), "Check ACOLITE_DIR and installation"
sys.path.insert(0, str(ACOLITE_DIR))
import acolite as ac
print("Input:", SAFE_PATH)
print("ACOLITE:", getattr(ac, "version", "installed"))

Input: /Users/rshunmugapandi/sentinel2_acolite_practical/input_l1c/S2C_MSIL1C_20260226T050801_N0512_R019_T43PFK_20260226T084516.SAFE
ACOLITE: Generic GitHub Clone c2026-07-17T15:09:19


## 8. Configure DSF atmospheric correction

The following settings apply **Dark Spectrum Fitting** and request NetCDF outputs needed for visual comparison. `limit` is supplied in **south, west, north, east** order in ACOLITE, unlike the AOI variable order above. Tiled aerosol estimation is an initial choice for spatially heterogeneous coastal environments.

The L1R `rhot_*` and L2R `rhos_*` arrays will be compared. This exercise deliberately does not enable cloud-removal or gap-filling. Bright clouds may be present in the images; avoid interpreting them as water reflectance.

**Version note:** ACOLITE options evolve; if your installed release rejects a setting, check the repository `config/defaults.txt` and the output log before adjusting it.

In [8]:
settings = {
    "inputfile": str(SAFE_PATH),
    "output": str(OUTPUT_DIR),
    "limit": [SOUTH, WEST, NORTH, EAST],
    "aerosol_correction": "dark_spectrum",
    "dsf_aot_estimate": "tiled",
    "s2_target_res": 20,
    "rgb_rhot": True,
    "rgb_rhos": True,
    "l1r_delete_netcdf": False,
    "l2r_delete_netcdf": False,
    "l2w_delete_netcdf": False,
}
print(settings)

{'inputfile': '/Users/rshunmugapandi/sentinel2_acolite_practical/input_l1c/S2C_MSIL1C_20260226T050801_N0512_R019_T43PFK_20260226T084516.SAFE', 'output': '/Users/rshunmugapandi/sentinel2_acolite_practical/acolite_output', 'limit': [8.25, 76.65, 8.75, 77.15], 'aerosol_correction': 'dark_spectrum', 'dsf_aot_estimate': 'tiled', 's2_target_res': 20, 'rgb_rhot': True, 'rgb_rhos': True, 'l1r_delete_netcdf': False, 'l2r_delete_netcdf': False, 'l2w_delete_netcdf': False}


## 9. Run ACOLITE

Execute the processing call once. The first run may spend time downloading atmospheric LUTs. A successful run should produce `*_L1R.nc` and `*_L2R.nc` (and potentially L2W, depending on your version/settings). Read the console output and saved logs if processing fails.

**Interpretation:** `L1R` is reprojected TOA reflectance, `L2R` is atmospherically corrected surface reflectance. For water-leaving products, examine the separately derived `L2W` outputs.

In [ ]:
result = ac.acolite.acolite_run(settings=settings)
print("Run returned:", result)
print("NetCDF outputs:")
for p in sorted(OUTPUT_DIR.rglob("*.nc")):
    print(p.name)

Running ACOLITE processing - Generic GitHub Clone c2026-07-17T15:09:19
Python - darwin - 3.13.14 | packaged by conda-forge | (main, Jun 12 2026, 09:44:24) [Clang 19.1.7 ]
Platform - Darwin 24.5.0 - arm64 - Darwin Kernel Version 24.5.0: Tue Apr 22 19:54:43 PDT 2025; root:xnu-11417.121.6~2/RELEASE_ARM64_T8132
Run ID - 20261012_132129
Identified /Users/rshunmugapandi/sentinel2_acolite_practical/input_l1c/S2C_MSIL1C_20260226T050801_N0512_R019_T43PFK_20260226T084516.SAFE as Sentinel-2 .SAFE type
Starting conversion of 1 scenes
Starting conversion of /Users/rshunmugapandi/sentinel2_acolite_practical/input_l1c/S2C_MSIL1C_20260226T050801_N0512_R019_T43PFK_20260226T084516.SAFE
Importing metadata from L1C_T43PFK_A007711_20260226T052617
Reading per pixel geometry
Computing band average per detector geometry
Detector 1
Detector 2
Detector 3


/Users/rshunmugapandi/acolite/acolite/sentinel2/l1_convert.py:398: RuntimeWarning: Mean of empty slice
  ave_vza = np.nanmean(ave_vza, axis=2)
/Users/rshunmugapandi/acolite/acolite/sentinel2/l1_convert.py:399: RuntimeWarning: Mean of empty slice
  ave_vaa = np.nanmean(ave_vaa, axis=2)


Detector 4
Detector 5
Detector 6
Wrote raa (2779, 1415)
Wrote vza (2779, 1415)
Wrote sza (2779, 1415)
Writing geolocation lon/lat
Wrote lon (2779, 1415)
Wrote lat (2779, 1415)
Converting bands
Converting bands: Wrote rhot_444 ((2779, 1415))
Converting bands: Wrote rhot_489 ((2779, 1415))
Converting bands: Wrote rhot_561 ((2779, 1415))
Converting bands: Wrote rhot_667 ((2779, 1415))
Converting bands: Wrote rhot_707 ((2779, 1415))
Converting bands: Wrote rhot_741 ((2779, 1415))
Converting bands: Wrote rhot_785 ((2779, 1415))
Converting bands: Wrote rhot_835 ((2779, 1415))
Converting bands: Wrote rhot_866 ((2779, 1415))
Converting bands: Wrote rhot_947 ((2779, 1415))
Converting bands: Wrote rhot_1372 ((2779, 1415))
Converting bands: Wrote rhot_1612 ((2779, 1415))
Converting bands: Wrote rhot_2191 ((2779, 1415))
Conversion took 5.3 seconds
Created /Users/rshunmugapandi/sentinel2_acolite_practical/acolite_output/S2C_MSI_2026_02_26_05_26_37_T43PFK_L1R.nc
Wrote /Users/rshunmugapandi/sentinel2

## 10. Open TOA and corrected reflectance files

This cell locates one `L1R` and one `L2R` output. ACOLITE encodes band wavelengths in dataset names, e.g. `rhot_...` and `rhos_...`. We select the nearest available **green band (about 560 nm)**. Matching approximate wavelengths is essential for a valid TOA/AC comparison.

In [ ]:
import numpy as np
import xarray as xr

l1_files = sorted(OUTPUT_DIR.rglob("*_L1R.nc"))
l2_files = sorted(OUTPUT_DIR.rglob("*_L2R.nc"))
if not l1_files or not l2_files:
    raise FileNotFoundError("ACOLITE L1R/L2R NetCDF files not found. Inspect the run log.")
l1 = xr.open_dataset(l1_files[-1])
l2 = xr.open_dataset(l2_files[-1])

def nearest_band(ds, prefix, wavelength=560):
    names = [k for k in ds.data_vars if k.startswith(prefix + "_")]
    if not names:
        raise KeyError(f"No {prefix}_* bands: {list(ds.data_vars)}")
    return min(names, key=lambda k: abs(float(k.split("_")[-1]) - wavelength))

toa_band = nearest_band(l1, "rhot")
ac_band = nearest_band(l2, "rhos")
toa = l1[toa_band].squeeze().load()
corrected = l2[ac_band].squeeze().load()
print("TOA:", toa_band, toa.shape, "AC:", ac_band, corrected.shape)

## 11. Plot TOA vs atmospherically corrected reflectance

Display the same band with a **shared colour scale**. This allows differences in magnitude to be compared directly. If the output grids have different shapes, this notebook stops rather than comparing mismatched pixels. Both arrays should have matching grids when generated from the same ACOLITE processing run.

The plots are **surface reflectance maps**, not photographs; differences arise from atmospheric correction and algorithm assumptions.

In [ ]:
import matplotlib.pyplot as plt

if toa.shape != corrected.shape:
    raise ValueError("L1R/L2R shapes differ; regrid before pixelwise comparison")
a = np.asarray(toa, dtype=float)
b = np.asarray(corrected, dtype=float)
valid = np.isfinite(a) & np.isfinite(b)
if not np.any(valid):
    raise ValueError("No shared valid pixels for comparison")
lo, hi = np.nanpercentile(np.concatenate([a[valid], b[valid]]), [2, 98])
fig, axs = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, arr, title in zip(axs, [a, b], [f"TOA ({toa_band})", f"AC / DSF ({ac_band})"]):
    im = ax.imshow(arr, cmap="viridis", vmin=lo, vmax=hi)
    ax.set_title(title)
    ax.set_xlabel("Pixel column"); ax.set_ylabel("Pixel row")
fig.colorbar(im, ax=axs, shrink=0.8, label="Reflectance (dimensionless)")
fig.savefig(FIG_DIR / "TOA_vs_ACOLITE_DSF_green.png", dpi=200)
plt.show()

## 12. Plot atmospheric-correction difference

Calculate `corrected − TOA` reflectance for each pixel and inspect its spatial pattern. A negative difference is common when atmospheric path reflectance is removed, but not guaranteed. **Do not infer correction accuracy from the sign alone.** This difference also includes effects of molecular absorption/scattering modelling and other steps.

In [ ]:
difference = np.where(valid, b - a, np.nan)
lim = np.nanpercentile(np.abs(difference), 98)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(difference, cmap="RdBu_r", vmin=-lim, vmax=lim)
ax.set(title="ACOLITE DSF − TOA (~560 nm)", xlabel="Pixel column", ylabel="Pixel row")
fig.colorbar(im, ax=ax, label="Difference in reflectance")
fig.tight_layout()
fig.savefig(FIG_DIR / "DSF_minus_TOA_green.png", dpi=200)
plt.show()
print("TOA median:", round(float(np.nanmedian(a[valid])), 5))
print("AC median:", round(float(np.nanmedian(b[valid])), 5))

## 13. Compare band-mean spectra (optional)

Compute a spatial median for each available TOA/AC band. **Caution:** This is descriptive, not formal spectral validation. Median values across all pixels may mix water, clouds and land and therefore are not an isolated water spectrum. For a publication-quality analysis, select a validated water-only ROI and exclude clouds, glint and land.

In [ ]:
def band_medians(ds, prefix):
    rows = []
    for name in ds.data_vars:
        if name.startswith(prefix + "_"):
            try:
                wave = float(name.split("_")[-1])
                value = float(ds[name].median(skipna=True).values)
                rows.append((wave, value))
            except (ValueError, TypeError):
                pass
    return np.array(sorted(rows))

toa_spectrum = band_medians(l1, "rhot")
ac_spectrum = band_medians(l2, "rhos")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(toa_spectrum[:, 0], toa_spectrum[:, 1], "o-", label="TOA (rhot)")
ax.plot(ac_spectrum[:, 0], ac_spectrum[:, 1], "s-", label="DSF corrected (rhos)")
ax.set(xlabel="Wavelength (nm)", ylabel="Spatial median reflectance", title="Scene-median spectra")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(FIG_DIR / "TOA_vs_DSF_spectra.png", dpi=200)
plt.show()

## 14. Report the final results

1. Name of the selected Sentinel-2 L1C `.SAFE` scene and acquisition date.
2. TOA (`rhot`) and atmospherically corrected (`rhos`) comparison for the same spectral band.
3. Difference map (`rhos − rhot`), with reflectance units and colour scale.
4. Brief interpretation of changed reflectance over the water region and visible cloud/land artifacts.


## References and troubleshooting

- [ACOLITE source, installation and methodology](https://github.com/acolite/acolite)
- [ACOLITE documentation](https://acolite.readthedocs.io/)
- [Copernicus Data Space OData catalogue and authenticated downloads](https://documentation.dataspace.copernicus.eu/APIs/OData.html)
- [Copernicus sample: catalogue search and download](https://documentation.dataspace.copernicus.eu/notebook-samples/geo/odata_basics.html)

**Common issues:**

- **No scenes found:** increase the date window or cloud percentage; verify the AOI latitude/longitude.
- **401/403 on download:** verify CDSE login and access-token authentication; the catalogue is public but product downloads require authorization.
- **ZIP extraction fails:** a download may be incomplete; delete that ZIP and download again.
- **No module named `acolite` / `osgeo`:** use the correct Conda kernel and install official dependencies.
- **No `L2R` output:** review ACOLITE logs, input product validity, disk space and LUT download connection.
- **Unexpected image differences:** confirm wavelength correspondence and check whether land/cloud pixels influence the scene statistics.

